# 4. Modelos base

Todos los modelos con hiperparámetros por defecto sobre la partición común. Líneas base:
`DummyRegressor` (la media) y la **persistencia** (el valor actual como predicción del siguiente
periodo), que en tasas de interés es la referencia exigente. La brecha entrenamiento–prueba
diagnostica sobreajuste (sección 1 del curso: < 10 % aceptable, > 15 % sobreajuste).

Las salidas son multivariables: cada modelo predice a la vez las 4 tasas (P1) o los 4 indicadores
(P2). Tiempo aproximado en una RTX 4050: P1 ~1 min, P2 ~5–10 min, P3 ~25 min (Prophet y ARIMA en CPU).

In [ ]:
import sys, warnings, json, time
from pathlib import Path
ML = Path.cwd().parent                      # carpeta ml/
sys.path.insert(0, str(ML))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
PROCESSED, REPORTS = ML / "data" / "processed", ML / "reports"
PRED = REPORTS / "predicciones"; PRED.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", palette="deep")

In [ ]:
# ===================== CONFIGURACIÓN =====================
USAR_GPU = True
CORRER = {"p1": True, "p2": True, "p3": True}
P3_HORIZONTES = [1, 3, 6, 12]

In [ ]:
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from mirage.ml import gpu as G
CFG = json.load(open(PROCESSED / "configuracion_particion.json"))
DEV = "cuda" if USAR_GPU else "cpu"
print("Dispositivo de los modelos:", DEV, "| GPU disponible:", G.device_disponible())

def preprocesador(num, cat):
    numerico = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())])
    categorico = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                           ("codificar", OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False))])
    return ColumnTransformer([("num", numerico, num)] + ([("cat", categorico, cat)] if cat else []))

def pipeline(est, num, cat):
    return Pipeline([("prep", preprocesador(num, cat)), ("modelo", est)])

def pliegues_temporales(periodos, n_splits=CFG["N_PLIEGUES_CV"]):
    u = np.sort(pd.Series(periodos).unique())
    return [(np.where(pd.Series(periodos).isin(u[a]))[0], np.where(pd.Series(periodos).isin(u[b]))[0])
            for a, b in TimeSeriesSplit(n_splits=min(n_splits, len(u) - 1)).split(u)]

def metricas(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float); nz = np.abs(y) > 1e-9
    return {"R2": r2_score(y, p), "MAE": mean_absolute_error(y, p), "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "MAPE_%": 100 * mean_absolute_percentage_error(y[nz], p[nz])}

def cargar(problema):
    """X, Y, periodo, columnas y máscara de entrenamiento de P1 o P2, con la partición guardada."""
    d = pd.read_parquet(PROCESSED / f"dataset_{problema}.parquet")
    d = d.merge(pd.read_parquet(PROCESSED / f"particion_{problema}.parquet"), on="id")
    if problema == "p1":
        Y = d[[c for c in d.columns if c.startswith("y_")]]; periodo = d["mes"]
        X = d.drop(columns=list(Y.columns) + ["split", "id", "mes"])
    else:
        Y = d[["y_de", "y_cob", "y_kd", "y_roa"]]; periodo = d["anio_objetivo"]
        X = d[["ciiu_division", "log_activos", "de", "de_l1", "cob", "kd", "roa", "roa_l1", "margen", "caja_activos",
               "deuda_activos", "crec_deuda", "tasa_impuesto_efectiva", "tasa_politica", "tes_10a_banrep", "ust_10a"]]
    cat = ["ciiu_division"]; num = [c for c in X.columns if c not in cat]
    return X, Y, periodo, num, cat, (d["split"] == "train").to_numpy(), d["id"].to_numpy()

def persistencia(problema, X, Y):
    fuente = {"p1": {c: c.replace("y_", "") + "_l0" for c in Y.columns},
              "p2": {"y_de": "de", "y_cob": "cob", "y_kd": "kd", "y_roa": "roa"}}[problema]
    return pd.DataFrame({c: X[s].to_numpy() for c, s in fuente.items()}).fillna(Y.median())

In [ ]:
# ===================== MODELOS (hiperparámetros por defecto) =====================
# Editar, quitar o agregar modelos aquí. "dispositivo" indica dónde corre cada uno.
MODELOS = {
    "Base (media)":       (DummyRegressor(strategy="mean"), "cpu"),
    "Regresión lineal":   (G.GPULinear(alpha=0.0, device=DEV), DEV),
    "Ridge":              (G.GPULinear(alpha=1.0, device=DEV), DEV),
    "Lasso":              (G.GPULasso(alpha=1e-3, device=DEV), DEV),
    "k-NN":               (G.GPUKNN(n_neighbors=5, device=DEV), DEV),
    "Árbol de decisión":  (G.xgb_tree(max_depth=8, device=DEV), DEV),
    "Random Forest":      (G.xgb_forest(n_estimators=100, max_depth=12, subsample=0.6, colsample_bynode=0.6, device=DEV), DEV),
    "Gradient Boosting":  (G.xgb_boost(n_estimators=300, max_depth=3, learning_rate=0.1, device=DEV), DEV),
    "XGBoost":            (G.xgb_boost(n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8, device=DEV), DEV),
    "SVR":                (G.GPUSVR(C=1.0, epsilon=0.1, n_components=2000, device=DEV), DEV),
    "Red neuronal (MLP)": (G.GPUMLP(hidden_layer_sizes=(64, 32), device=DEV), DEV),
}
pd.DataFrame({"modelo": list(MODELOS), "dispositivo": [v[1] for v in MODELOS.values()]})

In [ ]:
def correr(problema, modelos, etapa, mallas=None):
    X, Y, periodo, num, cat, tr, ids = cargar(problema)
    filas = []
    for nombre, (est, dev) in modelos.items():
        t0 = time.time(); pipe = pipeline(clone(est), num, cat); info = {}
        malla = (mallas or {}).get(nombre)
        if malla:
            gs = GridSearchCV(pipe, malla, cv=pliegues_temporales(periodo[tr].reset_index(drop=True)),
                              scoring="neg_mean_squared_error", n_jobs=1, refit=True)
            gs.fit(X[tr].reset_index(drop=True), Y[tr].reset_index(drop=True)); modelo = gs.best_estimator_
            info = {"mejores_parametros": {k: str(v) for k, v in gs.best_params_.items()}, "mse_cv": -gs.best_score_}
        else:
            modelo = pipe.fit(X[tr], Y[tr])
        seg = time.time() - t0
        P_tr = pd.DataFrame(np.asarray(modelo.predict(X[tr])).reshape(tr.sum(), -1), columns=Y.columns)
        P_te = pd.DataFrame(np.asarray(modelo.predict(X[~tr])).reshape((~tr).sum(), -1), columns=Y.columns)
        P_te.assign(id=ids[~tr]).to_parquet(PRED / f"{problema}_{etapa}_{nombre}.parquet", index=False)
        for c in Y.columns:
            filas.append({"modelo": nombre, "objetivo": c, "dispositivo": dev, "segundos": seg,
                          "R2_entrenamiento": r2_score(Y.loc[tr, c], P_tr[c]), **metricas(Y.loc[~tr, c], P_te[c]),
                          "parametros": json.dumps(info.get("mejores_parametros", {}), ensure_ascii=False)})
        print(f"{problema} · {nombre:20s} [{dev}] {seg:7.1f} s", flush=True)
    pers = persistencia(problema, X[~tr].reset_index(drop=True), Y[tr])
    pers.assign(id=ids[~tr]).to_parquet(PRED / f"{problema}_{etapa}_Persistencia.parquet", index=False)
    for c in Y.columns:
        filas.append({"modelo": "Persistencia", "objetivo": c, "dispositivo": "-", "segundos": 0, "R2_entrenamiento": np.nan,
                      **metricas(Y.loc[~tr, c], pers[c]), "parametros": "{}"})
    Y[~tr].assign(id=ids[~tr]).to_parquet(PRED / f"{problema}_real.parquet", index=False)
    res = pd.DataFrame(filas); res.to_csv(REPORTS / f"{problema}_{etapa}_metricas.csv", index=False)
    return res

def resumen(res):
    # Tolera tablas de corridas anteriores sin todas las columnas
    for c, v in (("dispositivo", "cpu"), ("segundos", np.nan), ("R2_entrenamiento", np.nan)):
        if c not in res: res[c] = v
    if "conjunto" in res:   # formato de la corrida con scripts: separar entrenamiento y prueba
        tr = res[res.conjunto == "entrenamiento"].set_index(["modelo", "objetivo"])["R2"]
        res = res[res.conjunto == "prueba"].copy(); res["R2_entrenamiento"] = res.set_index(["modelo", "objetivo"]).index.map(tr)
    t = res.groupby("modelo").agg(dispositivo=("dispositivo", "first"), segundos=("segundos", "first"), R2_entrenamiento=("R2_entrenamiento", "mean"),
                                  R2=("R2", "mean"), MAE=("MAE", "mean"), RMSE=("RMSE", "mean"))
    t["brecha_%"] = 100 * (t.R2_entrenamiento - t.R2) / t.R2_entrenamiento.abs()
    t["diagnostico"] = np.select([t["brecha_%"] < 10, t["brecha_%"] < 15], ["aceptable", "revisar"], "sobreajuste")
    return t.sort_values("RMSE").round(3)

## 4.1 P1 · Kd por sector y tamaño

In [ ]:
if CORRER["p1"]: r1 = correr("p1", MODELOS, "base")
r1 = pd.read_csv(REPORTS / "p1_base_metricas.csv"); resumen(r1)

In [ ]:
r1.pivot_table(index="modelo", columns="objetivo", values="RMSE").round(3).sort_values("y_tasa_gran")

## 4.2 P2 · Estructura de capital por empresa

In [ ]:
if CORRER["p2"]: r2 = correr("p2", MODELOS, "base")
r2 = pd.read_csv(REPORTS / "p2_base_metricas.csv"); resumen(r2)

In [ ]:
r2.pivot_table(index="modelo", columns="objetivo", values="R2").round(3).sort_values("y_de", ascending=False)

## 4.3 P3 · Insumos de mercado (origen móvil)

In [ ]:
P3 = pd.read_parquet(PROCESSED / "dataset_p3.parquet")
OBJ3 = ["spread_soberano", "tes_cop_10a", "tes_uvr_10a", "inflacion_implicita_10a", "tasa_politica", "ust_10a"]

In [ ]:
# ---------- Modelos de series de tiempo (CPU: no tienen versión en GPU) ----------
import logging; logging.getLogger("cmdstanpy").setLevel(logging.ERROR); logging.getLogger("prophet").setLevel(logging.ERROR)
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.api import VAR
from prophet import Prophet

def paseo_aleatorio(train, h, **kw): return train.iloc[-1]

def arima(train, h, ordenes=((1, 1, 0), (0, 1, 1), (1, 1, 1), (2, 1, 0)), **kw):
    out = {}
    for c in train.columns:
        mejor, aic = None, np.inf
        for o in ordenes:
            try:
                r = ARIMA(train[c].values, order=o).fit()
                if r.aic < aic: mejor, aic = r, r.aic
            except Exception: pass
        out[c] = mejor.forecast(h)[-1] if mejor is not None else train[c].iloc[-1]
    return pd.Series(out)

def var(train, h, maxlags=2, **kw):
    d = train.diff().dropna()
    try:
        r = VAR(d).fit(maxlags=maxlags, ic="aic"); k = max(r.k_ar, 1)
        return train.iloc[-1] + r.forecast(d.values[-k:], steps=h).sum(axis=0)
    except Exception:
        return train.iloc[-1]

def prophet(train, h, cps=0.05, **kw):
    out = {}
    for c in train.columns:
        m = Prophet(changepoint_prior_scale=cps, yearly_seasonality=True, weekly_seasonality=False,
                    daily_seasonality=False, uncertainty_samples=0).fit(pd.DataFrame({"ds": train.index, "y": train[c].values}))
        out[c] = float(m.predict(pd.DataFrame({"ds": [train.index[-1] + pd.offsets.MonthEnd(h)]}))["yhat"].iloc[0])
    return pd.Series(out)

def origen_movil(S, modelo, h, meses_prueba, minimo=48, **kw):
    """En cada mes t del periodo de prueba se reentrena con la historia hasta t y se pronostica t+h."""
    idx, filas = S.index, {}
    for t in idx[-(meses_prueba + h):-h]:
        tr = S.loc[:t]
        if len(tr) >= minimo: filas[idx[idx.get_loc(t) + h]] = modelo(tr, h, **kw)
    return pd.DataFrame(filas).T.sort_index()

def supervisado_p3(P3, h, rezagos=3):
    """Estrategia directa: con la información hasta t, predecir el vector de objetivos en t+h."""
    f = {}
    for c in OBJ3 + ["ibr_3m", "bei_usd_10a"]:
        for k in range(rezagos): f[f"{c}_l{k}"] = P3[c].shift(k)
        f[f"{c}_d1"] = P3[c].diff()
    f["colcap_ret"] = P3["colcap"].pct_change(fill_method=None); f["trm_ret"] = P3["trm"].pct_change(fill_method=None)
    X = pd.DataFrame(f, index=P3.index); Y = P3[OBJ3].shift(-h); ok = X.notna().all(1) & Y.notna().all(1)
    return X[ok], Y[ok]

def ml_origen_movil(P3, h, estimador, malla=None, meses_prueba=36):
    """Modelos de ML con rezagos, reentrenados en cada origen (y con GridSearchCV si se pasa malla)."""
    X, Y = supervisado_p3(P3, h); S = P3[OBJ3]; preds = {}
    for t in Y.index[-meses_prueba:]:
        pos = Y.index.get_loc(t); tr = Y.index[:max(pos - h + 1, 0)]   # solo filas cuyo objetivo ya se conoce en t
        if len(tr) < 36: continue
        pipe = pipeline(clone(estimador), list(X.columns), [])
        if malla: pipe = GridSearchCV(pipe, malla, cv=TimeSeriesSplit(n_splits=3), scoring="neg_mean_squared_error")
        pipe.fit(X.loc[tr], Y.loc[tr]); j = S.index.get_loc(t) + h
        if j < len(S): preds[S.index[j]] = pd.Series(np.ravel(pipe.predict(X.loc[[t]]))[:len(OBJ3)], index=OBJ3)
    return pd.DataFrame(preds).T.sort_index()

In [ ]:
def correr_p3(etapa, series_modelos, ml_modelos, mallas=None):
    S, filas = P3[OBJ3], []
    for h in P3_HORIZONTES:
        for nombre, (fn, kw) in series_modelos.items():
            t0 = time.time(); f = origen_movil(S, fn, h, CFG["P3_MESES_PRUEBA"], **kw); seg = time.time() - t0
            f.to_parquet(PRED / f"p3_{etapa}_h{h}_{nombre}.parquet")
            for c in OBJ3: filas.append({"modelo": nombre, "h": h, "objetivo": c, "dispositivo": "cpu", "segundos": seg, **metricas(S.loc[f.index, c], f[c])})
            print(f"h={h:2d} {nombre:16s} [cpu] {seg:6.1f} s", flush=True)
        for nombre, (est, dev) in ml_modelos.items():
            t0 = time.time(); f = ml_origen_movil(P3, h, est, (mallas or {}).get(nombre), CFG["P3_MESES_PRUEBA"]); seg = time.time() - t0
            f.to_parquet(PRED / f"p3_{etapa}_h{h}_{nombre}.parquet")
            for c in OBJ3: filas.append({"modelo": nombre, "h": h, "objetivo": c, "dispositivo": dev, "segundos": seg, **metricas(S.loc[f.index, c], f[c])})
            print(f"h={h:2d} {nombre:16s} [{dev}] {seg:6.1f} s", flush=True)
    res = pd.DataFrame(filas); res.to_csv(REPORTS / f"p3_{etapa}_metricas.csv", index=False); return res

SERIES = {"Paseo aleatorio": (paseo_aleatorio, {}), "ARIMA": (arima, {}), "VAR": (var, {"maxlags": 2}), "Prophet": (prophet, {"cps": 0.05})}
ML_P3 = {k: MODELOS[k] for k in ("Ridge", "Random Forest", "XGBoost")}
if CORRER["p3"]: r3 = correr_p3("base", SERIES, ML_P3)
r3 = pd.read_csv(REPORTS / "p3_base_metricas.csv")
r3.pivot_table(index="modelo", columns="h", values="RMSE").round(3).sort_values(1)

In [ ]:
ax = r3.pivot_table(index="modelo", columns="h", values="RMSE").T.plot(marker="o", figsize=(9, 4))
ax.set_xlabel("Horizonte (meses)"); ax.set_ylabel("RMSE promedio (pp)"); ax.set_title("P3: error por horizonte"); plt.show()
r3.pivot_table(index=["objetivo", "modelo"], columns="h", values="RMSE").round(3)